In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from copy import deepcopy
from tqdm.notebook import tqdm as tq
import matplotlib.pyplot as plt
%matplotlib inline

In [60]:
metadata = pd.read_csv('../../data/PRJEB93790.csv')

In [62]:
metadata = pd.read_csv('../../data/PRJEB93790.csv')
selected_cols = ['run_accession', 'host_subject_id', 'collection_timepoint',  
                 'host_sex', 'host_age', 'Householdno', 'Wheredoyoulive', 
                 'Maternaleducationalstatus',
                 'Ifyeswhenwasthelasttimetheygaveyousuchdewormingpill'] + \
                sorted([col for col in metadata.columns.values 
                          if (('sample_eggcount' in col) &
                              (~col.endswith('baseline')) &
                              (~col.endswith('followup') &
                               (~col.endswith('AnySTHs') & 
                               (~col.endswith('Others')))))]) + \
                ['sample_eggcountOthers']
metadata = metadata[selected_cols] 
metadata['STHStatus'] = ['+' if row[['sample_eggcountAL', 
                                    'sample_eggcountHW', 
                                    'sample_eggcountTT']].sum() > 0 else 
                         '-'
                        for i, row in metadata.iterrows()]
metadata.head()

,run_accession,host_subject_id,collection_timepoint,host_sex,host_age,Householdno,Wheredoyoulive,Maternaleducationalstatus,Ifyeswhenwasthelasttimetheygaveyousuchdewormingpill,sample_eggcountAL,sample_eggcountHW,sample_eggcountSM,sample_eggcountTT,sample_eggcountOthers,STHStatus
0,ERR15286707,399-KB,follow-up,female,14,5.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,-
1,ERR15286713,401-KB,follow-up,female,15,5.0,0.0,2.0,7Months,0.0,0.0,0.0,0.0,0.0,-
2,ERR15286719,29-KB,follow-up,female,10,4.0,0.0,NaN,7Months,0.0,0.0,0.0,0.0,0.0,-
3,ERR15286728,256A-KB,baseline,female,14,5.0,1.0,99.0,NaN,3.0,0.0,0.0,0.0,0.0,+
4,ERR15286730,131-KB,baseline,female,11,7.0,0.0,1.0,7Months,0.0,0.0,0.0,0.0,0.0,-


In [63]:
metadata_sample_name = pd.read_csv('../../data/PRJEB93790.csv')[['run_accession', 'sample_alias']]
metadata_sample_name.index = metadata_sample_name['sample_alias'].values

otu_data = pd.read_csv('../../data/phyloseq/nohost_asv/otu_table.csv', index_col=0)
tax_data = pd.read_csv('../../data/phyloseq/nohost_asv/tax_table.csv', index_col=0)
asv_di = {val:'ASV%d'%(i+1) for i, val in enumerate(tax_data.index.values)}
tax_data.index = [asv_di[val] for val in tax_data.index.values]
otu_data.columns = [asv_di[val] for val in otu_data.columns.values]
otu_data.index = metadata_sample_name.loc[otu_data.index.values, 'run_accession'].values
otu_data = otu_data.loc[metadata.run_accession.values]
metadata['total_read_counts'] = otu_data.sum(axis=1).values

otu_data = otu_data.apply(lambda x: x/x.sum(), axis=1)
otu_data.head()

,ASV1,ASV2,ASV3,ASV4,ASV5,ASV6,ASV7,ASV8,ASV9,ASV10,...,ASV17013,ASV17014,ASV17015,ASV17016,ASV17017,ASV17018,ASV17019,ASV17020,ASV17021,ASV17022
ERR15286707,0.033252,0.025214,0.000000,0.016461,0.000000,0.012548,0.018319,0.006367,0.016993,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ERR15286713,0.035055,0.025630,0.111758,0.049135,0.085558,0.036501,0.005783,0.011742,0.008835,0.016318,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ERR15286719,0.127891,0.096692,0.074305,0.036102,0.057315,0.028284,0.012041,0.003265,0.004805,0.008250,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ERR15286728,0.094521,0.059299,0.052063,0.020286,0.033293,0.013046,0.061755,0.037420,0.005407,0.013741,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ERR15286730,0.141804,0.089952,0.000000,0.049433,0.000000,0.030690,0.021106,0.031596,0.011076,0.009032,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
with pd.ExcelWriter('../../results/tables/Table S1. Sample metadata, relative abundance and taxonomy.xlsx') as writer:  
    metadata.to_excel(writer, sheet_name='Metadata', index=False)
    otu_data.transpose().to_excel(writer, sheet_name='Relative abundance')
    tax_data.to_excel(writer, sheet_name='Taxonomy')